# *Eidolon helvum* — iNaturalist occurrences × Earth Engine EVI and precipitation

Parallel to `dhhm_inat_pipeline.ipynb` (Death's-head hawkmoth), for the
straw-coloured fruit bat. Life stage is ignored. Workflow:

1. Fetch all verifiable iNaturalist observations (cached to CSV).
2. Clean and aggregate to **0.2° cell × day** records.
3. Sample "available" background cell-days (accessible land cells, same dates).
4. Extract from Google Earth Engine, for every cell-day and its four neighbours:
   * **EVI** (MODIS MOD13A2, 1 km, 16-day) at the observation date
   * **EVI temporal rate of change** — linear slope over the preceding 64 days (EVI per 30 days)
   * **Precipitation** (CHIRPS daily) — total over the preceding 30 days
   * **Precipitation temporal change** — last 30 days minus the 30 days before (mm per 30 days)
5. **Spatial rate of change across the landscape** — finite-difference gradient
   between the neighbouring cells (per 100 km) and the direction of increase.
6. Used vs available comparison, and monthly "green wave" maps / Hovmöller
   of EVI and rainfall change with the observations overlaid.

**Why 0.2° cells:** *E. helvum* is IUCN Near Threatened, so iNaturalist
obscures every public coordinate: the point is randomised within the
0.2° × 0.2° cell that contains the true location. Environmental values are
therefore averaged over that cell (which is guaranteed to contain the bat),
not sampled at the public point. With curator / trusted access to true
coordinates, set `CELL_DEG` smaller.

In [ ]:
# Setup — only needs installing once
# %pip install -q earthengine-api requests pandas numpy matplotlib scipy

import os
import json
import time


import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.path import Path as MplPath
from scipy import stats
from scipy.spatial import cKDTree

# ---- Config ----------------------------------------------------------------
TAXON_NAME    = "Eidolon helvum"
QUALITY       = ["research"]          # add "needs_id" to include unconfirmed records
CELL_DEG      = 0.2                   # iNaturalist obscuration cell for threatened taxa
EE_PROJECT    = "skilled-snow-435910-j9"
OUT_DIR       = os.path.join("results", "eidolon_helvum")
REFRESH_INAT  = False                 # True = re-download observations
BG_RATIO      = 3                     # background cell-days per observed cell-day
BG_RADIUS_KM  = 500                   # "accessible" area: land cells within this distance of any observation
EVI_SLOPE_DAYS = 64                   # window for the EVI temporal slope (≈ 4-5 composites)
EE_CHUNK      = 400                   # features per Earth Engine request
MAP_BBOX      = (-20.0, -36.0, 52.0, 22.0)   # lon_min, lat_min, lon_max, lat_max (sub-Saharan Africa + SW Arabia)
MAP_RES       = 0.1                   # degrees, climatology maps
CLIM_YEARS    = (2015, 2024)          # inclusive, climatology maps
SEED          = 42

os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
INAT_API = "https://api.inaturalist.org/v1"

## 1. Fetch iNaturalist observations

In [ ]:
raw_csv = os.path.join(OUT_DIR, "inat_observations_raw.csv")

def inat_controlled_terms():
    """{attribute_id: (label, {value_id: value_label})} for annotation decoding."""
    res = requests.get(f"{INAT_API}/controlled_terms", timeout=60).json()["results"]
    return {t["id"]: (t["label"], {v["id"]: v["label"] for v in t["values"]}) for t in res}

def parse_obs(o, terms):
    ann = {}
    for a in o.get("annotations", []):
        label, values = terms.get(a["controlled_attribute_id"], (None, {}))
        if label:
            ann[label] = values.get(a["controlled_value_id"])
    lon, lat = (o.get("geojson") or {}).get("coordinates", [None, None])
    return {
        "id": o["id"],
        "observed_on": o.get("observed_on"),
        "time_observed_at": o.get("time_observed_at"),
        "latitude": lat,
        "longitude": lon,
        "public_positional_accuracy": o.get("public_positional_accuracy"),
        "positional_accuracy": o.get("positional_accuracy"),
        "obscured": o.get("obscured"),
        "taxon_geoprivacy": o.get("taxon_geoprivacy"),
        "geoprivacy": o.get("geoprivacy"),
        "quality_grade": o.get("quality_grade"),
        "captive": o.get("captive"),
        "user_login": (o.get("user") or {}).get("login"),
        "place_guess": o.get("place_guess"),
        "life_stage": ann.get("Life Stage"),
        "alive_dead": ann.get("Alive or Dead"),
        "sex": ann.get("Sex"),
        "n_photos": len(o.get("photos", [])),
        "uri": o.get("uri"),
    }

def fetch_inat(taxon_name):
    taxon = requests.get(f"{INAT_API}/taxa", params={"q": taxon_name, "rank": "species"},
                         timeout=60).json()["results"][0]
    print(f"Taxon {taxon['id']}: {taxon['name']} ({taxon.get('preferred_common_name')})")
    terms = inat_controlled_terms()
    rows, id_above = [], 0
    # id_above paging avoids the API's 10,000-result page limit
    while True:
        r = requests.get(f"{INAT_API}/observations", timeout=60, params={
            "taxon_id": taxon["id"], "verifiable": "true", "per_page": 200,
            "order_by": "id", "order": "asc", "id_above": id_above})
        r.raise_for_status()
        res = r.json()["results"]
        if not res:
            break
        rows += [parse_obs(o, terms) for o in res]
        id_above = res[-1]["id"]
        print(f"  {len(rows)} observations", end="\r")
        time.sleep(1)   # iNaturalist asks for ≤ 1 request/s
    print()
    return pd.DataFrame(rows)

if REFRESH_INAT or not os.path.exists(raw_csv):
    raw = fetch_inat(TAXON_NAME)
    raw.to_csv(raw_csv, index=False)
else:
    raw = pd.read_csv(raw_csv)
print(f"{len(raw)} verifiable observations  ->  {raw_csv}")
print(raw["quality_grade"].value_counts().to_string())
print(f"Obscured: {raw['obscured'].mean():.0%}; median public accuracy "
      f"{raw['public_positional_accuracy'].median() / 1000:.0f} km")

## 2. Clean and aggregate to 0.2° cell × day

In [ ]:
obs = raw[raw["quality_grade"].isin(QUALITY) & (raw["captive"] != True)].copy()
obs["date"] = pd.to_datetime(obs["observed_on"], errors="coerce")
obs = obs.dropna(subset=["latitude", "longitude", "date"])
obs["year"]  = obs["date"].dt.year
obs["month"] = obs["date"].dt.month
obs["doy"]   = obs["date"].dt.dayofyear

# cell lower-left corner; the small epsilon keeps points on a cell edge in the cell above
obs["cell_lon"] = (np.floor(obs["longitude"] / CELL_DEG + 1e-9) * CELL_DEG).round(4)
obs["cell_lat"] = (np.floor(obs["latitude"]  / CELL_DEG + 1e-9) * CELL_DEG).round(4)
obs["cell_id"]  = obs["cell_lon"].map("{:.1f}".format) + "_" + obs["cell_lat"].map("{:.1f}".format)

# Several photos of the same roost on the same day are one record: collapse to
# cell-days (reduces pseudoreplication from colonies photographed repeatedly)
cell_days = (obs.groupby(["cell_id", "cell_lon", "cell_lat", "date"])
                .agg(n_obs=("id", "size"), n_users=("user_login", "nunique"),
                     dead=("alive_dead", lambda s: (s == "Dead").any()))
                .reset_index())
cell_days["year"]  = cell_days["date"].dt.year
cell_days["month"] = cell_days["date"].dt.month

print(f"{len(obs)} {'/'.join(QUALITY)} observations -> {len(cell_days)} cell-days "
      f"in {cell_days['cell_id'].nunique()} cells, {obs['year'].min()}-{obs['year'].max()}")

## 3. Descriptive maps and seasonality

In [ ]:
ne_path = os.path.join(OUT_DIR, "ne_50m_admin_0_countries.geojson")
if not os.path.exists(ne_path):
    url = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/"
           "geojson/ne_50m_admin_0_countries.geojson")
    with open(ne_path, "wb") as fh:
        fh.write(requests.get(url, timeout=120).content)
with open(ne_path, encoding="utf-8") as fh:
    countries = json.load(fh)

def country_rings(gj):
    """Exterior rings (Nx2 arrays) of all country polygons."""
    rings = []
    for f in gj["features"]:
        g = f["geometry"]
        polys = [g["coordinates"]] if g["type"] == "Polygon" else g["coordinates"]
        rings += [np.asarray(p[0]) for p in polys]
    return rings

RINGS = country_rings(countries)

def plot_countries(ax, face="0.92", edge="white", lw=0.4, bbox=MAP_BBOX):
    for r in RINGS:
        ax.fill(r[:, 0], r[:, 1], facecolor=face, edgecolor=edge, linewidth=lw, zorder=0)
    ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3]); ax.set_aspect("equal")

month_cmap = plt.get_cmap("twilight_shifted")   # cyclic: December sits next to January

fig, ax = plt.subplots(figsize=(9, 8))
plot_countries(ax)
sc = ax.scatter(cell_days["cell_lon"] + CELL_DEG / 2, cell_days["cell_lat"] + CELL_DEG / 2,
                c=cell_days["month"], cmap=month_cmap, vmin=0.5, vmax=12.5,
                s=10 + 4 * np.sqrt(cell_days["n_obs"]), alpha=0.8, edgecolor="k", linewidth=0.2)
cb = fig.colorbar(sc, ax=ax, shrink=0.6, ticks=range(1, 13)); cb.set_label("Month")
ax.set_title(f"{TAXON_NAME} — iNaturalist cell-days (n = {len(cell_days)}), coloured by month")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "map_cell_days_by_month.png"), dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
cell_days.groupby("year").size().plot.bar(ax=axes[0], color="steelblue")
axes[0].set(title="Cell-days per year", xlabel=None, ylabel="Cell-days")
# Migration signal: latitude of records through the year
axes[1].scatter(cell_days["date"].dt.dayofyear, cell_days["cell_lat"] + CELL_DEG / 2,
                s=8, alpha=0.4, color="k")
lat_by_month = cell_days.groupby("month")["cell_lat"].quantile([0.25, 0.5, 0.75]).unstack()
mid_doy = pd.to_datetime([f"2001-{m:02d}-15" for m in lat_by_month.index]).dayofyear
axes[1].plot(mid_doy, lat_by_month[0.5] + CELL_DEG / 2, color="crimson", lw=2, label="monthly median")
axes[1].fill_between(mid_doy, lat_by_month[0.25] + CELL_DEG / 2, lat_by_month[0.75] + CELL_DEG / 2,
                     color="crimson", alpha=0.2, label="IQR")
axes[1].set(title="Latitude of records through the year", xlabel="Day of year", ylabel="Latitude (°)")
axes[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "records_by_year_and_latitude.png"), dpi=200); plt.show()

## 4. Background ("available") cell-days

Land cells within `BG_RADIUS_KM` of any observed cell define the accessible
area. Background dates are resampled from the observed dates so used and
available share the same seasonal and inter-annual distribution.
iNaturalist records are biased towards people (cities, roads, colonies in
parks); uniform land background does not correct for that.

In [ ]:
def land_mask(lon, lat):
    pts = np.column_stack([lon, lat]); inside = np.zeros(len(pts), bool)
    for r in RINGS:
        if (r[:, 0].max() < MAP_BBOX[0] - 5) or (r[:, 0].min() > MAP_BBOX[2] + 5):
            continue
        inside |= MplPath(r).contains_points(pts)
    return inside

def to_xyz(lon, lat):
    lon, lat = np.radians(lon), np.radians(lat)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

grid_lon, grid_lat = np.meshgrid(np.arange(MAP_BBOX[0], MAP_BBOX[2], CELL_DEG),
                                 np.arange(MAP_BBOX[1], MAP_BBOX[3], CELL_DEG))
cand = pd.DataFrame({"cell_lon": grid_lon.ravel().round(4), "cell_lat": grid_lat.ravel().round(4)})
ctr_lon, ctr_lat = cand["cell_lon"] + CELL_DEG / 2, cand["cell_lat"] + CELL_DEG / 2
cand = cand[land_mask(ctr_lon, ctr_lat)]

# chord distance on the unit sphere for BG_RADIUS_KM
obs_cells = cell_days.drop_duplicates("cell_id")
tree  = cKDTree(to_xyz(obs_cells["cell_lon"] + CELL_DEG / 2, obs_cells["cell_lat"] + CELL_DEG / 2))
chord = 2 * np.sin(BG_RADIUS_KM / 6371.0 / 2)
d, _  = tree.query(to_xyz(cand["cell_lon"] + CELL_DEG / 2, cand["cell_lat"] + CELL_DEG / 2),
                   distance_upper_bound=chord)
accessible = cand[np.isfinite(d)].reset_index(drop=True)

n_bg = BG_RATIO * len(cell_days)
bg = accessible.sample(n=n_bg, replace=n_bg > len(accessible), random_state=SEED).reset_index(drop=True)
bg["date"] = rng.choice(cell_days["date"].values, size=n_bg, replace=True)
bg["cell_id"] = bg["cell_lon"].map("{:.1f}".format) + "_" + bg["cell_lat"].map("{:.1f}".format)
print(f"{len(accessible)} accessible land cells; {n_bg} background cell-days")

fig, ax = plt.subplots(figsize=(8, 7))
plot_countries(ax)
ax.scatter(accessible["cell_lon"], accessible["cell_lat"], s=0.3, color="tan", label="accessible")
ax.scatter(bg["cell_lon"], bg["cell_lat"], s=2, color="grey", alpha=0.5, label="background")
ax.scatter(obs_cells["cell_lon"], obs_cells["cell_lat"], s=4, color="crimson", label="observed")
ax.legend(loc="lower left", markerscale=4)
ax.set_title("Observed vs background cells")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "background_cells.png"), dpi=200); plt.show()

## 5. Earth Engine extraction

Authenticate once in a terminal with `earthengine authenticate` (or run
`ee.Authenticate()` here). Each cell-day gets five rectangles: its own cell
(C) and the neighbouring cells east, west, north and south, all reduced to
their mean on the same date. Results are cached per chunk, so an
interrupted run resumes where it stopped.

In [ ]:
import ee

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

MOD13  = ee.ImageCollection("MODIS/061/MOD13A2")
CHIRPS = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")
BANDS  = ["evi", "evi_slope_30d", "precip_30d", "precip_change_30d"]
OFFSETS = {"C": (0, 0), "E": (1, 0), "W": (-1, 0), "N": (0, 1), "S": (0, -1)}

# data availability: MODIS from 2000-02-18 (+ slope window), CHIRPS to its latest day
modis_start = pd.Timestamp("2000-02-18") + pd.Timedelta(days=EVI_SLOPE_DAYS)
chirps_end  = pd.Timestamp(ee.Date(CHIRPS.aggregate_max("system:time_start")).format("YYYY-MM-dd").getInfo())
modis_end   = pd.Timestamp(ee.Date(MOD13.aggregate_max("system:time_start")).format("YYYY-MM-dd").getInfo())
print(f"MODIS EVI composites to {modis_end.date()}, CHIRPS to {chirps_end.date()}")

def env_image(d):
    """EVI, EVI slope, 30-day rain and rain change for ee.Date d."""
    def prep(im):
        t = ee.Date(im.get("system:time_start")).difference(d, "day")
        return (ee.Image.constant(t).float().rename("t")
                .addBands(im.select("EVI").multiply(0.0001).rename("evi"))
                .copyProperties(im, ["system:time_start"]))
    evi_col   = MOD13.filterDate(d.advance(-EVI_SLOPE_DAYS, "day"), d.advance(1, "day")).map(prep)
    evi_now   = ee.Image(evi_col.sort("system:time_start", False).first()).select("evi")
    # linearFit takes [x, y]; scale = EVI per day -> per 30 days
    evi_slope = evi_col.select(["t", "evi"]).reduce(ee.Reducer.linearFit()).select("scale") \
                       .multiply(30).rename("evi_slope_30d")
    p30   = CHIRPS.filterDate(d.advance(-30, "day"), d).sum().rename("precip_30d")
    pprev = CHIRPS.filterDate(d.advance(-60, "day"), d.advance(-30, "day")).sum()
    return ee.Image.cat([evi_now, evi_slope, p30, p30.subtract(pprev).rename("precip_change_30d")])

samples = pd.concat([cell_days.assign(type="observed")[["cell_id", "cell_lon", "cell_lat", "date", "type"]],
                     bg.assign(type="background")[["cell_id", "cell_lon", "cell_lat", "date", "type"]]],
                    ignore_index=True)
samples["date"] = pd.to_datetime(samples["date"])
ok = (samples["date"] >= modis_start) & (samples["date"] <= min(chirps_end, modis_end))
print(f"Dropping {(~ok).sum()} cell-days outside the EVI / CHIRPS record")
samples = samples[ok].copy()
samples["date_str"] = samples["date"].dt.strftime("%Y-%m-%d")
samples["key"] = samples["type"].str[0] + "_" + samples["cell_id"] + "_" + samples["date_str"]
samples = samples.drop_duplicates("key").reset_index(drop=True)

rects = pd.concat([samples.assign(pos=p, x0=samples["cell_lon"] + dx * CELL_DEG,
                                  y0=samples["cell_lat"] + dy * CELL_DEG)
                   for p, (dx, dy) in OFFSETS.items()], ignore_index=True)

def extract_chunk(chunk):
    feats = [ee.Feature(ee.Geometry.Rectangle([r.x0, r.y0, r.x0 + CELL_DEG, r.y0 + CELL_DEG], None, False),
                        {"key": r.key, "pos": r.pos, "date": r.date_str})
             for r in chunk.itertuples()]
    fc = ee.FeatureCollection(feats)
    # one environmental image per date, reduced over every rectangle on that date
    def per_date(ds):
        return env_image(ee.Date(ds)).reduceRegions(
            fc.filter(ee.Filter.eq("date", ds)), ee.Reducer.mean(), scale=1000)
    out = ee.FeatureCollection(ee.List(sorted(chunk["date_str"].unique())).map(per_date)).flatten()
    out = out.select(["key", "pos"] + BANDS, None, False)   # drop geometries from the payload
    return pd.DataFrame([f["properties"] for f in out.getInfo()["features"]])

env_csv = os.path.join(OUT_DIR, "ee_env_extraction_long.csv")
done = pd.read_csv(env_csv) if os.path.exists(env_csv) else pd.DataFrame(columns=["key", "pos"])
todo = rects[~(rects["key"] + rects["pos"]).isin(done["key"] + done["pos"])].sort_values("date_str")
print(f"{len(rects)} rectangles, {len(todo)} still to extract")

for i in range(0, len(todo), EE_CHUNK):
    chunk = todo.iloc[i:i + EE_CHUNK]
    for attempt in range(3):
        try:
            res = extract_chunk(chunk); break
        except ee.EEException as e:          # transient "computation timed out" / quota errors
            print(f"  chunk {i // EE_CHUNK}: {str(e)[:120]} — retry {attempt + 1}")
            time.sleep(10 * (attempt + 1))
    else:
        raise RuntimeError("Earth Engine extraction failed three times; re-run to resume")
    res.to_csv(env_csv, mode="a", header=not os.path.exists(env_csv), index=False)
    print(f"  {min(i + EE_CHUNK, len(todo))}/{len(todo)}", end="\r")
print()

## 6. Spatial gradients and the analysis table

In [ ]:
env_long = pd.read_csv(env_csv).drop_duplicates(["key", "pos"])
env_wide = env_long.pivot(index="key", columns="pos", values=BANDS)
env_wide.columns = [f"{b}_{p}" for b, p in env_wide.columns]
env = samples.merge(env_wide.reset_index(), on="key", how="left")

def add_gradients(df, bands, cell_deg=CELL_DEG):
    """Central-difference gradient between neighbouring cells, per 100 km, and
    the compass bearing (deg) in which the variable increases."""
    lat_c = np.radians(df["cell_lat"] + cell_deg / 2)
    w_km  = cell_deg * 111.32 * np.cos(lat_c)     # cell width
    h_km  = cell_deg * 110.57                     # cell height
    for b in bands:
        gx = (df[f"{b}_E"] - df[f"{b}_W"]) / (2 * w_km) * 100
        gy = (df[f"{b}_N"] - df[f"{b}_S"]) / (2 * h_km) * 100
        df[f"{b}_grad_100km"] = np.hypot(gx, gy)
        df[f"{b}_grad_bearing"] = np.degrees(np.arctan2(gx, gy)) % 360
    return df

env = add_gradients(env, BANDS)
env["used"] = (env["type"] == "observed").astype(int)
env.to_csv(os.path.join(OUT_DIR, "eidolon_env_cell_days.csv"), index=False)

ANALYSIS_VARS = {
    "evi_C":                    "EVI",
    "evi_slope_30d_C":          "EVI change (per 30 d)",
    "precip_30d_C":             "Rain, last 30 d (mm)",
    "precip_change_30d_C":      "Rain change vs previous 30 d (mm)",
    "evi_grad_100km":           "EVI gradient (per 100 km)",
    "precip_30d_grad_100km":    "Rain gradient (mm per 100 km)",
}
print(env.groupby("type")[list(ANALYSIS_VARS)].median().T.round(3))

## 7. Used vs available

In [ ]:
def logistic_irls(X, y, n_iter=50):
    """Logistic regression by IRLS; returns coefficients, SE, z, p (with intercept)."""
    X = np.column_stack([np.ones(len(X)), X]); beta = np.zeros(X.shape[1])
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-X @ beta)); W = p * (1 - p)
        H = X.T @ (X * W[:, None])
        step = np.linalg.solve(H, X.T @ (y - p)); beta += step
        if np.max(np.abs(step)) < 1e-8:
            break
    se = np.sqrt(np.diag(np.linalg.inv(H))); z = beta / se
    return pd.DataFrame({"coef": beta, "se": se, "z": z, "p": 2 * stats.norm.sf(np.abs(z))})

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
tests = []
for ax, (col, label) in zip(axes.ravel(), ANALYSIS_VARS.items()):
    u, a = env.loc[env["used"] == 1, col].dropna(), env.loc[env["used"] == 0, col].dropna()
    lo, hi = np.nanpercentile(env[col], [1, 99])
    bins = np.linspace(lo, hi, 40)
    ax.hist(a, bins=bins, density=True, alpha=0.5, color="grey", label=f"available (n={len(a)})")
    ax.hist(u, bins=bins, density=True, alpha=0.6, color="crimson", label=f"used (n={len(u)})")
    ax.set(title=label, xlabel=None); ax.legend(fontsize=8)
    mw = stats.mannwhitneyu(u, a)
    tests.append({"variable": label, "median_used": u.median(), "median_available": a.median(),
                  "mannwhitney_p": mw.pvalue,
                  # effect size in [-1, 1]; positive = used cell-days have higher values
                  "rank_biserial": 2 * mw.statistic / (len(u) * len(a)) - 1})
plt.suptitle(f"{TAXON_NAME}: environment at used vs available cell-days")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "used_vs_available.png"), dpi=200); plt.show()
print(pd.DataFrame(tests).round(4).to_string(index=False))

# Multivariable: standardised predictors, so coefficients are comparable
model_df = env.dropna(subset=list(ANALYSIS_VARS))
Z = (model_df[list(ANALYSIS_VARS)] - model_df[list(ANALYSIS_VARS)].mean()) / model_df[list(ANALYSIS_VARS)].std()
fit = logistic_irls(Z.values, model_df["used"].values)
fit.index = ["intercept"] + list(ANALYSIS_VARS.values())
print(f"\nLogistic regression, used (1) vs available (0), n = {len(model_df)}")
print(fit.round(4).to_string())

## 8. Monthly climatology: the "green wave" and rains

Monthly mean EVI and CHIRPS rainfall over `CLIM_YEARS`, and their month-to-
month rate of change (central difference, per month), with each month's
observations overlaid. The Hovmöller summarises the same thing by latitude.

In [ ]:
lon0, lat0, lon1, lat1 = MAP_BBOX
W, H = int(round((lon1 - lon0) / MAP_RES)), int(round((lat1 - lat0) / MAP_RES))
GRID = {"dimensions": {"width": W, "height": H},
        "affineTransform": {"scaleX": MAP_RES, "shearX": 0, "translateX": lon0,
                            "shearY": 0, "scaleY": -MAP_RES, "translateY": lat1},
        "crsCode": "EPSG:4326"}

def ee_to_array(img):
    arr = ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY", "grid": GRID})
    return np.asarray(arr[arr.dtype.names[0]], dtype=float)

clim_npz = os.path.join(OUT_DIR, f"climatology_{CLIM_YEARS[0]}_{CLIM_YEARS[1]}.npz")
if os.path.exists(clim_npz):
    clim = np.load(clim_npz); evi_m, rain_m = clim["evi"], clim["rain"]
else:
    y0, y1 = f"{CLIM_YEARS[0]}-01-01", f"{CLIM_YEARS[1] + 1}-01-01"
    n_years = CLIM_YEARS[1] - CLIM_YEARS[0] + 1
    evi_m, rain_m = [], []
    for m in range(1, 13):
        cal = ee.Filter.calendarRange(m, m, "month")
        evi_m.append(ee_to_array(MOD13.filterDate(y0, y1).filter(cal).select("EVI").mean()
                                 .multiply(0.0001).unmask(np.nan)))
        rain_m.append(ee_to_array(CHIRPS.filterDate(y0, y1).filter(cal).sum()
                                  .divide(n_years).unmask(np.nan)))
        print(f"  month {m}", end="\r")
    evi_m, rain_m = np.stack(evi_m), np.stack(rain_m)
    np.savez_compressed(clim_npz, evi=evi_m, rain=rain_m)

# month-to-month rate of change, cyclic central difference (per month)
d_evi  = (np.roll(evi_m, -1, axis=0) - np.roll(evi_m, 1, axis=0)) / 2
d_rain = (np.roll(rain_m, -1, axis=0) - np.roll(rain_m, 1, axis=0)) / 2
extent = [lon0, lon1, lat0, lat1]
month_names = pd.date_range("2001-01-01", periods=12, freq="MS").strftime("%b")

def monthly_panels(field, cmap, vlim, label, fname):
    fig, axes = plt.subplots(3, 4, figsize=(16, 10.5), gridspec_kw={"hspace": 0.15, "wspace": 0.05})
    norm = mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)
    for m, ax in enumerate(axes.ravel()):
        im = ax.imshow(field[m], extent=extent, origin="upper", cmap=cmap, norm=norm)
        for r in RINGS:
            ax.plot(r[:, 0], r[:, 1], color="k", lw=0.2)
        cd = cell_days[cell_days["month"] == m + 1]
        ax.scatter(cd["cell_lon"] + CELL_DEG / 2, cd["cell_lat"] + CELL_DEG / 2,
                   s=6, facecolor="none", edgecolor="k", linewidth=0.5)
        ax.set_xlim(lon0, lon1); ax.set_ylim(lat0, lat1)
        ax.set_title(f"{month_names[m]} (n = {len(cd)})"); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.5, label=label)
    fig.suptitle(f"{TAXON_NAME} cell-days over {label}, {CLIM_YEARS[0]}-{CLIM_YEARS[1]}", y=0.92)
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=200, bbox_inches="tight"); plt.show()

monthly_panels(d_evi,  "BrBG", 0.08, "EVI change per month",   "monthly_evi_change.png")
monthly_panels(d_rain, "RdBu", 80,   "rain change per month (mm)", "monthly_rain_change.png")

# Hovmöller: zonal mean over the longitudes with records, by 1° latitude band
lon_idx = (np.arange(W) * MAP_RES + lon0 + MAP_RES / 2)
lon_sel = (lon_idx >= cell_days["cell_lon"].quantile(0.05)) & (lon_idx <= cell_days["cell_lon"].quantile(0.95))
lat_centres = lat1 - (np.arange(H) + 0.5) * MAP_RES
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, field, cmap, vlim, label in [(axes[0], d_evi, "BrBG", 0.05, "EVI change per month"),
                                     (axes[1], d_rain, "RdBu", 60, "Rain change per month (mm)")]:
    zonal = np.nanmean(field[:, :, lon_sel], axis=2).T          # (lat, month)
    im = ax.pcolormesh(np.arange(1, 13), lat_centres, zonal, cmap=cmap,
                       norm=mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim), shading="nearest")
    ax.plot(lat_by_month.index, lat_by_month[0.5] + CELL_DEG / 2, color="k", lw=2, label="records: median lat")
    ax.fill_between(lat_by_month.index, lat_by_month[0.25] + CELL_DEG / 2,
                    lat_by_month[0.75] + CELL_DEG / 2, color="k", alpha=0.15, label="IQR")
    ax.set(xticks=range(1, 13), xticklabels=month_names, title=label)
    fig.colorbar(im, ax=ax)
axes[0].set_ylabel("Latitude (°)"); axes[0].legend(loc="lower left")
fig.suptitle(f"Zonal mean ({lon_idx[lon_sel].min():.0f}° to {lon_idx[lon_sel].max():.0f}°E) vs record latitude")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "hovmoller_evi_rain_change.png"), dpi=200); plt.show()